In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent))

from  lora_transfer_pruning.core.pruning_instrumentor import PruningInstrumentor
from transformers import AutoModelForCausalLM
import torch
import compare_utils
from compare_utils import debug_group_prune_step_by_step
from lora_transfer_pruning.adapter.torch_pruning_group_builder import TorchPruningGroupBuilder
from compare_utils import full_attention_test_with_prune
from transformers import AutoConfig, AutoModelForCausalLM


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


In [2]:
MODEL = "google/gemma-4-E4B-it" 
DEVICE = "cuda:3"
def load_model(device=DEVICE):
    config = AutoConfig.from_pretrained(MODEL)

    text = config.text_config
    text.enable_moe_block = True
    text.num_experts = 4
    text.top_k_experts = 2
    text.moe_intermediate_size = 512

    # Для дешёвого теста.
    text.num_hidden_layers = 4

    model = AutoModelForCausalLM.from_config(
        config,
        dtype=torch.bfloat16,
    )

    return model.to(DEVICE)

In [3]:
model = load_model()

In [4]:
model

Gemma4ForConditionalGeneration(
  (model): Gemma4Model(
    (language_model): Gemma4TextModel(
      (embed_tokens): Gemma4TextScaledWordEmbedding(262144, 2560, padding_idx=0)
      (layers): ModuleList(
        (0-3): 4 x Gemma4TextDecoderLayer(
          (self_attn): Gemma4TextAttention(
            (q_norm): Gemma4RMSNorm()
            (k_norm): Gemma4RMSNorm()
            (v_norm): Gemma4RMSNorm()
            (k_proj): Linear(in_features=2560, out_features=512, bias=False)
            (q_proj): Linear(in_features=2560, out_features=2048, bias=False)
            (v_proj): Linear(in_features=2560, out_features=512, bias=False)
            (o_proj): Linear(in_features=2048, out_features=2560, bias=False)
          )
          (mlp): Gemma4TextMLP(
            (gate_proj): Linear(in_features=2560, out_features=10240, bias=False)
            (up_proj): Linear(in_features=2560, out_features=10240, bias=False)
            (down_proj): Linear(in_features=10240, out_features=2560, bias=Fals

In [5]:
from transformer_lens.model_bridge import TransformerBridge
import transformer_lens

bridge = TransformerBridge.boot_transformers(
    MODEL,
    hf_model=model,
    dtype=torch.float16,
)

In [6]:
bridge


TransformerBridge(
  (vision_encoder): GeneralizedComponent(
    (hook_in): HookPoint(name='vision_encoder.hook_in')
    (hook_out): HookPoint(name='vision_encoder.hook_out')
    (_original_component): Gemma4VisionModel(
      (patch_embedder): Gemma4VisionPatchEmbedder(
        (input_proj): Linear(in_features=768, out_features=768, bias=False)
      )
      (encoder): Gemma4VisionEncoder(
        (rotary_emb): Gemma4VisionRotaryEmbedding()
        (layers): ModuleList(
          (0-15): 16 x Gemma4VisionEncoderLayer(
            (self_attn): Gemma4VisionAttention(
              (q_proj): Gemma4ClippableLinear(
                (linear): Linear(in_features=768, out_features=768, bias=False)
              )
              (k_proj): Gemma4ClippableLinear(
                (linear): Linear(in_features=768, out_features=768, bias=False)
              )
              (v_proj): Gemma4ClippableLinear(
                (linear): Linear(in_features=768, out_features=768, bias=False)
              

In [7]:
bridge.blocks[0].experts

GeneralizedComponent(
  (hook_in): HookPoint(name='blocks.0.experts.hook_in')
  (hook_out): HookPoint(name='blocks.0.experts.hook_out')
  (_original_component): Gemma4TextExperts(
    (act_fn): GELUTanh()
  )
)

In [8]:
model.model.language_model.layers[0].self_attn.is_kv_shared_layer

False

In [9]:
model.model.language_model.layers[0].self_attn.kv_shared_layer_index

In [10]:
model.model.language_model.layers[0].self_attn.store_full_length_kv

False

In [11]:
model.model.language_model.layers[0].self_attn.use_alternative_attention

False

In [12]:
model.model.language_model.layers[0].self_attn.o_proj._original_component.bias

In [13]:
model.model.language_model.layers[0].self_attn._original_component.config.attention_bias

False

In [14]:
model.model.language_model.layers[0].self_attn._original_component.config._attn_implementation

'sdpa'

In [15]:
model.model.language_model.layers[3].self_attn._original_component.config.use_double_wide_mlp

False

In [16]:
model.model.language_model.layers[0]._original_component.enable_moe_block


True

In [17]:
model.model.language_model.layers[0]._original_component.hidden_size_per_layer_input

256

основной поток 2560 ───────────────────────────────┐
                                                   + → 2560
основной поток 2560 → gate 256                     │
                              × per-layer input 256 │
                              → projection 2560 ────┘

There is another branch of per_layer_input in forward of model.

Builded from special per-layer embeddings. Or from projection of main embedding (???)

In [18]:
# token t:
    # main embedding/hidden state: 2560
    # layer 0 extra input:          256
    # layer 1 extra input:          256
    # ...
    # layer 41 extra input:         256

In [19]:
model.model.language_model.layers[0]._original_component.hidden_size

2560

In [20]:
model.model.language_model.layers[0]._original_component.config.hidden_activation

'gelu_pytorch_tanh'

In [21]:
from datasets import load_dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL)
validation_dataset = load_dataset(
    "Salesforce/wikitext",
    "wikitext-2-raw-v1",
    split="validation",
)
validation_dataset

Dataset({
    features: ['text'],
    num_rows: 3760
})

In [22]:
for i, text in enumerate(validation_dataset):
    print(f"{i}: {text}")
    if i > 10:
        break

0: {'text': ''}
1: {'text': ' = Homarus gammarus = \n'}
2: {'text': ''}
3: {'text': ' Homarus gammarus , known as the European lobster or common lobster , is a species of clawed lobster from the eastern Atlantic Ocean , Mediterranean Sea and parts of the Black Sea . It is closely related to the American lobster , H. americanus . It may grow to a length of 60 cm ( 24 in ) and a mass of 6 kilograms ( 13 lb ) , and bears a conspicuous pair of claws . In life , the lobsters are blue , only becoming " lobster red " on cooking . Mating occurs in the summer , producing eggs which are carried by the females for up to a year before hatching into planktonic larvae . Homarus gammarus is a highly esteemed food , and is widely caught using lobster pots , mostly around the British Isles . \n'}
4: {'text': ''}
5: {'text': ' = = Description = = \n'}
6: {'text': ''}
7: {'text': ' Homarus gammarus is a large crustacean , with a body length up to 60 centimetres ( 24 in ) and weighing up to 5 – 6 kilogram

In [23]:
CONTEXT_LENGTH = 256
NUM_EVAL_BLOCKS = 32 #(block=batch)
EVAL_BATCH_SIZE = 1

validation_text = "\n\n".join(
    text for text in validation_dataset["text"] if text.strip()
)
validation_tokens = tokenizer(
    validation_text,
    add_special_tokens=False,
    return_tensors="pt",
).input_ids[0]

num_blocks = NUM_EVAL_BLOCKS
assert num_blocks > 0, "Validation split does not contain enough tokens"
evaluation_blocks = validation_tokens[: num_blocks * CONTEXT_LENGTH].reshape(
    num_blocks, CONTEXT_LENGTH
)
evaluation_blocks.shape

torch.Size([32, 256])

In [24]:
ignored_params = []
# for name, param in model.named_parameters():
#     if "norm" in name:
#         ignored_params.append(param)

In [25]:
import torch
import torch.nn as nn
import torch_pruning as tp

example_inputs = evaluation_blocks[:4].to(DEVICE)

def trace_forward(model, input_ids):
    return model(
        input=input_ids,
        use_cache=False,
        return_type="logits",
        #return_dict=True,
    ) #for compatability with tp

DG = tp.DependencyGraph().build_dependency(
    bridge,
    example_inputs=example_inputs,
    forward_fn=trace_forward,
    ignored_params=ignored_params,
        unwrapped_parameters=[(bridge.blocks[0].attn.q_norm._original_component.weight, 0),
                              (bridge.blocks[0].experts.gate_up_proj, 1),
                              (bridge.blocks[0].experts.down_proj, 2)
                              ]
)


/glazkov-dev/LoRa-Transfer-Pruning/.venv/lib/python3.10/site-packages/torch_pruning/dependency/graph.py:390: UserWarning: Unwrapped parameters detected: ['model.language_model.layers.1._original_component.experts._original_component.down_proj', 'model.audio_tower.layers.6.feed_forward2.ffw_layer_2.linear.weight', 'model.vision_tower._original_component.encoder.layers.6.pre_feedforward_layernorm.weight', 'model.audio_tower.layers.0.feed_forward2.pre_layer_norm.weight', 'model.audio_tower.layers.6.self_attn.k_proj.linear.weight', 'model.audio_tower.layers.9.self_attn.post.linear.weight', 'model.language_model.layers.0._original_component.post_feedforward_layernorm._original_component.weight', 'model.vision_tower._original_component.encoder.layers.9.self_attn.q_norm.weight', 'model.language_model.layers.2._original_component.router._original_component.per_expert_scale', 'model.audio_tower.layers.5.lconv1d.linear_start.linear.weight', 'model.audio_tower.layers.9.feed_forward1.ffw_layer_1.l

n.type check
n.type check
n.type check
n.type check
n.type check
n.type check
n.type check
n.type check
n.type check
n.type check
n.type check
n.type check
n.type check
n.type check
n.type check
n.type check
n.type check
n.type check
n.type check
n.type check
n.type check
n.type check
n.type check
n.type check
n.type check
n.type check
n.type check
n.type check
n.type check
n.type check
n.type check
n.type check
n.type check
n.type check
n.type check
n.type check
n.type check
n.type check
n.type check
n.type check
n.type check
n.type check
n.type check
n.type check
n.type check
n.type check
n.type check
n.type check
n.type check
n.type check
n.type check
n.type check
n.type check
n.type check
n.type check
n.type check
n.type check
n.type check
n.type check
n.type check
n.type check
n.type check
n.type check
n.type check


In [26]:
hasattr(bridge, 'rotary_emb')

True

In [27]:
#name of module, cols (in), rows(out)


#local configuration
d = {"blocks.0.attn.q": (None, [2, 6, 9]), #repeat indices for qkvo
     "blocks.0.mlp.up_proj": (None, [1, 3, 5])}

In [28]:
print(bridge.blocks[0]._original_component.enable_moe_block)
print(bridge.blocks[0].experts._original_component)

True
Gemma4TextExperts(
  (act_fn): GELUTanh()
)


In [68]:
group = DG.get_pruning_group(
    bridge.blocks[0].experts.gate_up_proj, 
    tp.prune_parameter_out_channels, 
    idxs=[2, 6, 9] )

In [69]:
print(group)


--------------------------------
          Pruning Group
--------------------------------
[0] prune_out_channels on UnwrappedParameter_1 (torch.Size([4, 1024, 2560])) => prune_out_channels on UnwrappedParameter_1 (torch.Size([4, 1024, 2560])), len(idxs)=3
[1] prune_out_channels on UnwrappedParameter_1 (torch.Size([4, 1024, 2560])) => prune_out_channels on _ElementWiseOp_507(TransposeBackward0), len(idxs)=3
[2] prune_out_channels on _ElementWiseOp_507(TransposeBackward0) => prune_out_channels on _ElementWiseOp_505(GroupedMmBackward0), len(idxs)=3
[3] prune_out_channels on _ElementWiseOp_505(GroupedMmBackward0) => prune_out_channels on _ElementWiseOp_506(IndexBackward0), len(idxs)=3
[4] prune_out_channels on _ElementWiseOp_505(GroupedMmBackward0) => prune_out_channels on _SplitOp_504([0]), len(idxs)=3
[5] prune_out_channels on _SplitOp_504([0]) => prune_out_channels on _ElementWiseOp_501(MulBackward0), len(idxs)=3
[6] prune_out_channels on _SplitOp_504([0]) => prune_out_channels on _Ele

So... torch pruning cant correctly process 3D tensors and unwrapped parameters, it cant handle dependencies correctly. Needed dependency: prune 2I dimension of gate_up_proj => prune I dim in down_proj, that's all.

In [51]:
#unofficial name (experts) in the bridge
#no official name path in transformer_lens
bridge.blocks[0].experts._original_component

Gemma4TextExperts(
  (act_fn): GELUTanh()
)

In [ ]:
print(hasattr(bridge.blocks[0].experts._original_component, "gate_up_proj"))
print(hasattr(bridge.blocks[0].experts._original_component, "down_proj"))
#there is no bridge wrapper for that parameters...
#the operations wrote inside as nn.functional.linear(x, gate_up_proj[expert_idx])

True
True


In [33]:
def manually_indices_repeating(num_heads: int, head_dim: int, pruning_indices: torch.Tensor):
    all_indices = []
    for head_num in range(num_heads):
        all_indices.append(
            pruning_indices+head_num*head_dim)
    return torch.cat(all_indices)

In [34]:
type(bridge.blocks[0].attn)

transformer_lens.model_bridge.generalized_components.base.GeneralizedComponent

In [35]:
bridge

TransformerBridge(
  (vision_encoder): GeneralizedComponent(
    (hook_in): HookPoint(name='vision_encoder.hook_in')
    (hook_out): HookPoint(name='vision_encoder.hook_out')
    (_original_component): Gemma4VisionModel(
      (patch_embedder): Gemma4VisionPatchEmbedder(
        (input_proj): Linear(in_features=768, out_features=768, bias=False)
      )
      (encoder): Gemma4VisionEncoder(
        (rotary_emb): Gemma4VisionRotaryEmbedding()
        (layers): ModuleList(
          (0-15): 16 x Gemma4VisionEncoderLayer(
            (self_attn): Gemma4VisionAttention(
              (q_proj): Gemma4ClippableLinear(
                (linear): Linear(in_features=768, out_features=768, bias=False)
              )
              (k_proj): Gemma4ClippableLinear(
                (linear): Linear(in_features=768, out_features=768, bias=False)
              )
              (v_proj): Gemma4ClippableLinear(
                (linear): Linear(in_features=768, out_features=768, bias=False)
              

In [36]:
#matches with config from llama
print(bridge.blocks[0].attn._original_component.config.num_attention_heads)
print(bridge.blocks[0].attn._original_component.config.head_dim)
print(bridge.blocks[0].attn._original_component.config.num_key_value_heads)

8
256
2


In [37]:
print(bridge.blocks[0].attn.q._original_component.weight.shape)
print(bridge.blocks[0].attn.k._original_component.weight.shape)


torch.Size([2048, 2560])
torch.Size([512, 2560])


In [38]:
bridge.blocks[0].attn._original_component.config.hidden_size

2560

In [39]:
bridge.blocks[0].attn.head_dim

256

In [40]:
repeated_idxs = manually_indices_repeating(
    bridge.blocks[0].attn._original_component.config.num_attention_heads,
    bridge.blocks[0].attn._original_component.config.head_dim,
    torch.tensor([2, 6, 9])
)
repeated_idxs

tensor([   2,    6,    9,  258,  262,  265,  514,  518,  521,  770,  774,  777,
        1026, 1030, 1033, 1282, 1286, 1289, 1538, 1542, 1545, 1794, 1798, 1801])

In [41]:
repeated_idxs.shape

torch.Size([24])

In [42]:
group = DG.get_pruning_group(
    bridge.blocks[0].attn.q._original_component, 
    tp.prune_linear_out_channels, 
    idxs=repeated_idxs.tolist() )

In [43]:
a = torch.as_tensor([[1, 2, 3], [4, 5, 6]])

In [44]:
a.view(-1)[0].item()

1

In [45]:
a.size(-1)
a * torch.full(a.shape[:-1], a.size(-1)).unsqueeze(-1)

tensor([[ 3,  6,  9],
        [12, 15, 18]])

In [46]:
print(group)


--------------------------------
          Pruning Group
--------------------------------
[0] prune_out_channels on blocks.0._original_component.self_attn._original_component.q_proj._original_component (Linear(in_features=2560, out_features=2048, bias=False)) => prune_out_channels on blocks.0._original_component.self_attn._original_component.q_proj._original_component (Linear(in_features=2560, out_features=2048, bias=False)), len(idxs)=24
[1] prune_out_channels on blocks.0._original_component.self_attn._original_component.q_proj._original_component (Linear(in_features=2560, out_features=2048, bias=False)) => prune_out_channels on _Reshape_3747(), len(idxs)=24
[2] prune_out_channels on _Reshape_3747() => prune_out_channels on _ElementWiseOp_3742(ToCopyBackward0), len(idxs)=24
[3] prune_out_channels on _ElementWiseOp_3742(ToCopyBackward0) => prune_out_channels on _ElementWiseOp_3740(MulBackward0), len(idxs)=24
[4] prune_out_channels on _ElementWiseOp_3742(ToCopyBackward0) => prune_out_c

In [47]:
bridge.get_submodule("model.language_model.layers.0._original_component.self_attn._original_component.q_norm")

GeneralizedComponent(
  (hook_in): HookPoint(name='blocks.0.attn.q_norm.hook_in')
  (hook_out): HookPoint(name='blocks.0.attn.q_norm.hook_out')
  (_original_component): Gemma4RMSNorm()
)

In [48]:
for i, (dep, idx) in enumerate(group):
    if (isinstance(dep.target.module, nn.Parameter)):
        print(dep.target.name)
        print("q norm", dep.target.module is bridge.blocks[0].attn.q_norm.weight)
        print("k norm", dep.target.module is bridge.blocks[0].attn.k_norm.weight)
        print(dep.handler)
        print(idx)
        print("param to name", group._DG._param_to_name[dep.target.module])
        # print(dep.target.module)

UnwrappedParameter_664 (torch.Size([256]))
q norm False
k norm True
<bound method ParameterPruner.prune_out_channels of <torch_pruning.pruner.function.ParameterPruner object at 0x720a98ef2c50>>
[2, 6, 9, 258, 262, 265, 514, 518, 521, 770, 774, 777, 1026, 1030, 1033, 1282, 1286, 1289, 1538, 1542, 1545, 1794, 1798, 1801]
param to name model.language_model.layers.0._original_component.self_attn._original_component.k_norm._original_component.weight
UnwrappedParameter_0 (torch.Size([256]))
q norm True
k norm False
<bound method ParameterPruner.prune_out_channels of <torch_pruning.pruner.function.ParameterPruner object at 0x720a98ef2c50>>
[2, 6, 9, 258, 262, 265, 514, 518, 521, 770, 774, 777, 1026, 1030, 1033, 1282, 1286, 1289, 1538, 1542, 1545, 1794, 1798, 1801]
param to name model.language_model.layers.0._original_component.self_attn._original_component.q_norm._original_component.weight


In [49]:
from transformer_lens.model_bridge.generalized_components.base import GeneralizedComponent


for name, mod in bridge.named_modules():
    if (isinstance(mod, TransformerBridge)):
        continue
    have = False
    if ("rms" in mod.__class__.__name__.lower()):
        have = True
    comp = False
    if (isinstance(mod, GeneralizedComponent)):
        comp = True
    print(name, have, comp)

vision_encoder False True
vision_encoder.hook_in False False
vision_encoder.hook_out False False
vision_encoder._original_component False False
vision_encoder._original_component.patch_embedder False False
vision_encoder._original_component.patch_embedder.input_proj False False
vision_encoder._original_component.encoder False False
vision_encoder._original_component.encoder.rotary_emb False False
vision_encoder._original_component.encoder.layers False False
vision_encoder._original_component.encoder.layers.0 False False
vision_encoder._original_component.encoder.layers.0.self_attn False False
vision_encoder._original_component.encoder.layers.0.self_attn.q_proj False False
vision_encoder._original_component.encoder.layers.0.self_attn.q_proj.linear False False
vision_encoder._original_component.encoder.layers.0.self_attn.k_proj False False
vision_encoder._original_component.encoder.layers.0.self_attn.k_proj.linear False False
vision_encoder._original_component.encoder.layers.0.self_attn.

Why only q_norm and k_norm found? Because v_norm without .weight.

So unwrapped RMS norms founded and added to deps in group automatically.

In [50]:
bridge.get_submodule("blocks.0._original_component.self_attn._original_component.k_proj")

LinearBridge(2560 -> 512, bias=False, original_component=Linear)

### Without MOE block, just mlp

In [57]:
bridge.blocks[0]._original_component.enable_moe_block

False

In [ ]:
from transformers.models.gemma4.modeling_gemma4 import Gemma4TextMLP
group = DG.get_pruning_group(
    bridge.blocks[0].mlp.up_proj._original_component, 
    tp.prune_linear_out_channels, 
    idxs=repeated_idxs.tolist() )

In [61]:
print(group)


--------------------------------
          Pruning Group
--------------------------------
[0] prune_out_channels on blocks.0._original_component.mlp._original_component.up_proj._original_component (Linear(in_features=2560, out_features=10240, bias=False)) => prune_out_channels on blocks.0._original_component.mlp._original_component.up_proj._original_component (Linear(in_features=2560, out_features=10240, bias=False)), len(idxs)=24
[1] prune_out_channels on blocks.0._original_component.mlp._original_component.up_proj._original_component (Linear(in_features=2560, out_features=10240, bias=False)) => prune_out_channels on _ElementWiseOp_3660(MulBackward0), len(idxs)=24
[2] prune_out_channels on _ElementWiseOp_3660(MulBackward0) => prune_out_channels on _ElementWiseOp_3661(GeluBackward0), len(idxs)=24
[3] prune_out_channels on _ElementWiseOp_3660(MulBackward0) => prune_out_channels on _Reshape_3658(), len(idxs)=24
[4] prune_out_channels on _Reshape_3658() => prune_out_channels on _ElementW

In [64]:
bridge.blocks[0].mlp._original_component.act_fn

GELUTanh()

### Fraction-based pruning comparison

Restart the kernel and run the model/dataset setup cells, but skip the preceding explicit-index pruning cell. This experiment samples channel indices from fractions, records the actual independently sampled RoPE coordinates for each attention layer, then compares activation and structural pruning using the same groups.

In [ ]:
# Fraction-based version of the activation-vs-structural comparison.
# Run on a freshly loaded, unpruned bridge; skip the preceding explicit-index
# comparison cell after restarting the kernel.
from compare_utils import compare_tp_and_transfer_pruning
from compare_utils import create_prune_task

FRACTION_ATTN_LAYERS = [0, 8, 16]
FRACTION_MLP_LAYERS = [4, 12, 20] 
#ATTN_OUT_FRACTION = 0.1
#MLP_OUT_FRACTION = 0.3 
ATTN_OUT_FRACTION = [1, 2, 99] #q proj
MLP_OUT_FRACTION = [3, 5, 1000] #up proj
FRACTION_SEED = 0

prune_task = create_prune_task(FRACTION_ATTN_LAYERS, FRACTION_MLP_LAYERS, ATTN_OUT_FRACTION, MLP_OUT_FRACTION)

So we see a little divergence in more fraction sizes. But it very close.

In [49]:
bridge.blocks.__len__()

42

In [50]:
compare_tp_and_transfer_pruning(bridge,
                                FRACTION_ATTN_LAYERS,
                                FRACTION_MLP_LAYERS,
                                ATTN_OUT_FRACTION,
                                MLP_OUT_FRACTION,
                                FRACTION_SEED,
                                evaluation_blocks,
                                EVAL_BATCH_SIZE
                                )

fraction prune_task:
  blocks.0.attn.q: (None, [1, 2])
  blocks.8.attn.q: (None, [1, 2])
  blocks.16.attn.q: (None, [1, 2])
  blocks.4.mlp.up_proj: (None, [3, 5])
  blocks.12.mlp.up_proj: (None, [3, 5])
  blocks.20.mlp.up_proj: (None, [3, 5])


/glazkov-dev/LoRa-Transfer-Pruning/.venv/lib/python3.10/site-packages/torch_pruning/dependency/graph.py:390: UserWarning: Unwrapped parameters detected: ['model.vision_tower._original_component.encoder.layers.9.post_attention_layernorm.weight', 'model.vision_tower._original_component.encoder.layers.13.self_attn.q_norm.weight', 'model.audio_tower.layers.6.feed_forward2.pre_layer_norm.weight', 'model.audio_tower.layers.6.lconv1d.linear_start.linear.weight', 'model.audio_tower.layers.9.feed_forward1.post_layer_norm.weight', 'model.language_model.layers.2._original_component.post_per_layer_input_norm._original_component.weight', 'model.language_model.layers.8._original_component.post_per_layer_input_norm._original_component.weight', 'model.language_model.layers.13._original_component.pre_feedforward_layernorm._original_component.weight', 'model.language_model.layers.22._original_component.post_per_layer_input_norm._original_component.weight', 'model.language_model.layers.28._original_compo

removed root indices by module:
  blocks.0.attn.q: count=32, idxs=[1, 2, 129, 130, 257, 258, 385, 386, 513, 514, 641, 642, 769, 770, 897, 898, 1025, 1026, 1153, 1154, 1281, 1282, 1409, 1410, 1537, 1538, 1665, 1666, 1793, 1794, 1921, 1922]
  blocks.8.attn.q: count=32, idxs=[1, 2, 129, 130, 257, 258, 385, 386, 513, 514, 641, 642, 769, 770, 897, 898, 1025, 1026, 1153, 1154, 1281, 1282, 1409, 1410, 1537, 1538, 1665, 1666, 1793, 1794, 1921, 1922]
  blocks.16.attn.q: count=32, idxs=[1, 2, 129, 130, 257, 258, 385, 386, 513, 514, 641, 642, 769, 770, 897, 898, 1025, 1026, 1153, 1154, 1281, 1282, 1409, 1410, 1537, 1538, 1665, 1666, 1793, 1794, 1921, 1922]
  blocks.4.mlp.up_proj: count=2, idxs=[3, 5]
  blocks.12.mlp.up_proj: count=2, idxs=[3, 5]
  blocks.20.mlp.up_proj: count=2, idxs=[3, 5]
layer=0: removed local head dims=4 (1.562%), idxs=[1, 2, 129, 130]
layer=8: removed local head dims=4 (1.562%), idxs=[1, 2, 129, 130]
layer=16: removed local head dims=4 (1.562%), idxs=[1, 2, 129, 130]
fractio

{'baseline': {'loss': 8.498046875, 'perplexity': 4905.17919921875},
 'transfer_activation': {'loss': 8.5166015625, 'perplexity': 4997.04296875},
 'torch_pruning_structural': {'loss': 8.5068359375,
  'perplexity': 4948.48095703125},
 'structural_minus_transfer': {'loss': -0.009765625,
  'perplexity': -48.56201171875}}

In [ ]:
# import importlib
# importlib.reload(compare_utils)

<module 'compare_utils' from '/glazkov-dev/LoRa-Transfer-Pruning/experiments/compare_tp_and_our/compare_utils.py'>

In [51]:
group = DG.get_pruning_group(
    bridge.blocks[0].attn.q._original_component, 
    tp.prune_linear_out_channels, 
    idxs=repeated_idxs.tolist() ) #IMPORTANT use tolist())

In [ ]:
repeated_idxs.__len__() # x4 more than kv   

24

In [49]:
gb = TorchPruningGroupBuilder(bridge, evaluation_blocks[:1].to(DEVICE))

/glazkov-dev/LoRa-Transfer-Pruning/.venv/lib/python3.10/site-packages/torch_pruning/dependency/graph.py:390: UserWarning: Unwrapped parameters detected: ['model.language_model.layers.0._original_component.self_attn._original_component.k_norm._original_component.weight', 'model.audio_tower.layers.2.norm_post_attn.weight', 'model.audio_tower.layers.4.self_attn.per_dim_scale', 'model.language_model.layers.12._original_component.input_layernorm._original_component.weight', 'model.language_model.layers.14._original_component.post_feedforward_layernorm._original_component.weight', 'model.language_model.layers.15._original_component.post_attention_layernorm._original_component.weight', 'model.language_model.layers.18._original_component.post_attention_layernorm._original_component.weight', 'model.language_model.layers.24._original_component.self_attn._original_component.k_norm._original_component.weight', 'model.language_model.layers.26._original_component.post_feedforward_layernorm._original

In [ ]:

correct_group = gb.get_correct_pruning_group( bridge.blocks[0].attn.q, 
    tp.prune_linear_out_channels,
    torch.tensor([2, 3, 5, 7]))

/glazkov-dev/LoRa-Transfer-Pruning/.venv/lib/python3.10/site-packages/torch_pruning/dependency/graph.py:390: UserWarning: Unwrapped parameters detected: ['model.language_model.layers.40._original_component.pre_feedforward_layernorm._original_component.weight', 'model.vision_tower._original_component.encoder.layers.7.post_feedforward_layernorm.weight', 'model.audio_tower.layers.3.lconv1d.pre_layer_norm.weight', 'model.language_model.layers.11._original_component.input_layernorm._original_component.weight', 'model.vision_tower._original_component.patch_embedder.position_embedding_table', 'model.audio_tower.subsample_conv_projection.layer1.norm.weight', 'model.audio_tower.layers.7.lconv1d.linear_end.linear.weight', 'model.audio_tower.layers.10.lconv1d.linear_end.linear.weight', 'model.audio_tower.layers.4.lconv1d.linear_end.linear.weight', 'model.language_model.layers.31._original_component.post_attention_layernorm._original_component.weight', 'model.language_model.layers.34._original_com

In [ ]:

debug_group_prune_step_by_step(correct_group)

LINEARS BEFORE {130010024354208: (2560, 2048, (2048, 2560), 130010006074800), 130010024353680: (2048, 2560, (2560, 2048), 130010006061680), 130010024354544: (2560, 512, (512, 2560), 130010006069040), 130010024354160: (2560, 512, (512, 2560), 130010006067520)}

[0] prune_out_channels target=blocks.0._original_component.self_attn._original_component.q_proj._original_component (Linear(in_features=2560, out_features=2048, bias=False)) idxs=64
CHANGED: (2560, 2048, (2048, 2560), 130010006074800) -> (2560, 1984, (1984, 2560), 129985625119600)

[1] prune_out_channels target=_Reshape_3747() idxs=64

[2] prune_out_channels target=_ElementWiseOp_3742(ToCopyBackward0) idxs=64

[3] prune_out_channels target=_ElementWiseOp_3740(MulBackward0) idxs=64

[4] prune_out_channels target=_ElementWiseOp_3746(PowBackward0) idxs=64

[5] prune_out_channels target=_ElementWiseOp_3745(MeanBackward1) idxs=64

[6] prune_out_channels target=_ElementWiseOp_3744(AddBackward0) idxs=64

[7] prune_out_channels target=_E

{130010024354208: (2560, 1984, (1984, 2560), 129985625119600),
 130010024353680: (1984, 2560, (2560, 1984), 129985625124800),
 130010024354544: (2560, 496, (496, 2560), 129985625119520),
 130010024354160: (2560, 496, (496, 2560), 129985625118960)}

In [50]:
correct_group = gb.get_correct_pruning_group( bridge.blocks[0].mlp.up_proj, 
    tp.prune_linear_out_channels,
    torch.tensor([2, 3, 5, 7]))

In [51]:
debug_group_prune_step_by_step(correct_group)

LINEARS BEFORE {124435119586704: (2560, 10240, (10240, 2560), 124435071110736), 124435119585072: (10240, 2560, (2560, 10240), 124435072836176), 124435119587184: (2560, 10240, (10240, 2560), 124435117785712)}

[0] prune_out_channels target=blocks.0._original_component.mlp._original_component.up_proj._original_component (Linear(in_features=2560, out_features=10240, bias=False)) idxs=4
CHANGED: (2560, 10240, (10240, 2560), 124435071110736) -> (2560, 10236, (10236, 2560), 124435082481472)

[1] prune_out_channels target=_ElementWiseOp_3660(MulBackward0) idxs=4

[2] prune_out_channels target=_ElementWiseOp_3661(GeluBackward0) idxs=4

[3] prune_out_channels target=_Reshape_3658() idxs=4

[4] prune_out_channels target=_ElementWiseOp_3657(MmBackward0) idxs=4

[5] prune_out_channels target=_ElementWiseOp_3659(TBackward0) idxs=4

[6] prune_in_channels target=blocks.0._original_component.mlp._original_component.down_proj._original_component (Linear(in_features=10240, out_features=2560, bias=False)

{124435119586704: (2560, 10236, (10236, 2560), 124435082481472),
 124435119585072: (10236, 2560, (2560, 10236), 124435082481232),
 124435119587184: (2560, 10236, (10236, 2560), 124435082483072)}